In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
import scipy.stats as stats
from scipy.stats import beta as beta_dist
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests

In [3]:
plt.style.use("seaborn-v0_8-whitegrid")
colors = {"control": "#4C72B0", "treatment": "#DD8452", "neutral": "#55A868"}
seed = 42
np.random.seed(seed)

In [4]:
df_ab = pd.read_csv("ab_test.csv")
df_countries = pd.read_csv("countries_ab.csv")


In [5]:
df_ab.head()

,id,time,con_treat,page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [6]:
df_countries.head()

,id,country
0,834778,UK
1,928468,US
2,822059,UK
3,711597,UK
4,710616,UK


In [7]:
df_ab['id'].isin(df_countries['id']).sum()

np.int64(294478)

In [8]:
df = pd.merge(df_ab, df_countries, on='id')
df.head()

,id,time,con_treat,page,converted,country
0,851104,11:48.6,control,old_page,0,US
1,804228,01:45.2,control,old_page,0,US
2,661590,55:06.2,treatment,new_page,0,US
3,853541,28:03.1,treatment,new_page,0,US
4,864975,52:26.2,control,old_page,1,US


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         294478 non-null  int64 
 1   time       294478 non-null  object
 2   con_treat  294478 non-null  object
 3   page       294478 non-null  object
 4   converted  294478 non-null  int64 
 5   country    294478 non-null  object
dtypes: int64(2), object(4)
memory usage: 13.5+ MB


In [10]:
def info_data(df):
    print("Shape of the dataset:", df.shape)
    print("\nData types:\n", df.dtypes)
    print("\nMissing values:\n", df.isnull().sum())
    print("\nUnique values:\n", df.nunique())

In [11]:
info_data(df)

Shape of the dataset: (294478, 6)

Data types:
 id            int64
time         object
con_treat    object
page         object
converted     int64
country      object
dtype: object

Missing values:
 id           0
time         0
con_treat    0
page         0
converted    0
country      0
dtype: int64

Unique values:
 id           290584
time          35993
con_treat         2
page              2
converted         2
country           3
dtype: int64


In [12]:
# Preprocessing
df['time'] = pd.to_timedelta('00:' + df['time'].astype(str))

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype          
---  ------     --------------   -----          
 0   id         294478 non-null  int64          
 1   time       294478 non-null  timedelta64[ns]
 2   con_treat  294478 non-null  object         
 3   page       294478 non-null  object         
 4   converted  294478 non-null  int64          
 5   country    294478 non-null  object         
dtypes: int64(2), object(3), timedelta64[ns](1)
memory usage: 13.5+ MB


In [14]:
df.groupby('id')['con_treat'].sum().value_counts()

con_treat
treatment             143397
control               143293
controlcontrol          1007
treatmenttreatment       992
treatmentcontrol         963
controltreatment         932
Name: count, dtype: int64

In [ ]:
df['con_treat'].sum().value_counts().plot(kind='bar')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

def load_and_validate(df: pd.DataFrame) -> pd.DataFrame:
    """Audit integrity based on columns: id, con_treat, converted."""
    print("=" * 60 + "\nSTEP 1: DATA QUALITY AUDIT\n" + "=" * 60)
    
    # --- 1. CLEAN THE LABELS FIRST ---
    # Combine doubles and mark mixed values as 'neither'
    conditions = [
        (df['con_treat'].isin(['control', 'controlcontrol'])),
        (df['con_treat'].isin(['treatment', 'treatmenttreatment']))
    ]
    choices = ['control', 'treatment']
    df['group_clean'] = np.select(conditions, choices, default='neither')
    
    # --- 2. USER CONTAMINATION CHECK ---
    # Check if an ID is associated with more than one CLEANED group
    multi_group = df.groupby("id")["group_clean"].nunique()
    contaminated = multi_group[multi_group > 1]
    
    if len(contaminated) > 0:
        print(f"⚠ Users in multiple groups: {len(contaminated)}. Removing...")
        df = df[~df["id"].isin(contaminated.index)].copy()
    
    # --- 3. SAMPLE RATIO MISMATCH (SRM) CHECK ---
    # We only run SRM on 'control' vs 'treatment' (excluding 'neither')
    analysis_df = df[df['group_clean'].isin(['control', 'treatment'])]
    group_counts = analysis_df["group_clean"].value_counts()
    
    if len(group_counts) == 2:
        _, srm_pval = stats.chisquare(group_counts.values)
        srm_flag = "⚠ SRM DETECTED" if srm_pval < 0.05 else "✓ No SRM"
        print(f"SRM p-value: {srm_pval:.4f}  {srm_flag}")
    
    print(f"\nFinal Group sizes (including 'neither'):\n{df['group_clean'].value_counts()}")
    
    return df

# Execute
df_validated = load_and_validate(df)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. EXPERIMENT DESIGN
# ─────────────────────────────────────────────────────────────────────────────
def experiment_design_doc(baseline=0.12, mde=0.02, alpha=0.05, power=0.8):
    print("\n" + "=" * 60 + "\nSTEP 2: EXPERIMENT DESIGN\n" + "=" * 60)
    # Effect size using arcsin transformation
    effect_size = 2 * (np.arcsin(np.sqrt(baseline + mde)) - np.arcsin(np.sqrt(baseline)))
    n_per_group = NormalIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=power)
    
    design = {"Required N per group": int(np.ceil(n_per_group)), "Alpha": alpha, "Power": power}
    for k, v in design.items(): print(f"  {k}: {v}")
    return design

experiment_design_doc(baseline=0.12, mde=0.02, alpha=0.05, power=0.8)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. FREQUENTIST TESTING (Updated for 'con_treat' and 'converted')
# ─────────────────────────────────────────────────────────────────────────────
def frequentist_analysis(df: pd.DataFrame):
    print("\n" + "=" * 60 + "\nSTEP 3: FREQUENTIST ANALYSIS\n" + "=" * 60)
    ctrl = df[df["con_treat"] == "control"]["converted"]
    treat = df[df["con_treat"] == "treatment"]["converted"]
    
    z_stat, p_val = proportions_ztest([treat.sum(), ctrl.sum()], [len(treat), len(ctrl)])
    
    results = pd.DataFrame({
        "group": ["control", "treatment"],
        "conv_rate": [ctrl.mean(), treat.mean()],
        "n": [len(ctrl), len(treat)]
    })
    print(results.to_string(index=False))
    print(f"\nP-value: {p_val:.6f} | {'✓ Significant' if p_val < 0.05 else '✗ Not Significant'}")
    return results
frequentist_results = frequentist_analysis(df_validated)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. BAYESIAN A/B TESTING
# ─────────────────────────────────────────────────────────────────────────────
def bayesian_analysis(df: pd.DataFrame, n_samples=100_000):
    print("\n" + "=" * 60 + "\nSTEP 4: BAYESIAN ANALYSIS\n" + "=" * 60)
    ctrl = df[df["con_treat"] == "control"]["converted"]
    treat = df[df["con_treat"] == "treatment"]["converted"]
    
    # Posteriors using Beta distribution
    alpha_c, beta_c = 1 + ctrl.sum(), 1 + (len(ctrl) - ctrl.sum())
    alpha_t, beta_t = 1 + treat.sum(), 1 + (len(treat) - treat.sum())
    
    samples_c = beta_dist.rvs(alpha_c, beta_c, size=n_samples)
    samples_t = beta_dist.rvs(alpha_t, beta_t, size=n_samples)
    
    prob_t_wins = np.mean(samples_t > samples_c)
    print(f"P(Treatment > Control): {prob_t_wins:.2%}")
    return {"prob_treatment_wins": prob_t_wins}

bayesian_results = bayesian_analysis(df_validated)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. SEQUENTIAL TESTING (O'Brien-Fleming)
# ─────────────────────────────────────────────────────────────────────────────
def sequential_analysis(df: pd.DataFrame):
    print("\n" + "=" * 60 + "\nSTEP 5: SEQUENTIAL ANALYSIS\n" + "=" * 60)
    # Sorting by 'time' column shown in image
    df_sorted = df.sort_values("time").reset_index(drop=True)
    z_alpha = stats.norm.ppf(1 - 0.05 / 2)
    
    for frac in [0.25, 0.5, 0.75, 1.0]:
        sub = df_sorted.head(int(len(df_sorted) * frac))
        c = sub[sub["con_treat"] == "control"]["converted"]
        t = sub[sub["con_treat"] == "treatment"]["converted"]
        z, _ = proportions_ztest([t.sum(), c.sum()], [len(t), len(c)])
        boundary = z_alpha / np.sqrt(frac)
        status = "STOP: Reject H₀" if abs(z) > boundary else "Continue"
        print(f" Look at {frac*100:.0f}%: |Z|={abs(z):.2f}, Boundary={boundary:.2f} -> {status}")
sequential_analysis(df_validated)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. MULTIPLE TESTING CORRECTION
# ─────────────────────────────────────────────────────────────────────────────
def multiple_testing_correction(df: pd.DataFrame):
    print("\n" + "=" * 60 + "\nSTEP 6: MULTIPLE TESTING\n" + "=" * 60)
    # Example checking 'converted' and 'page' as potential independent metrics
    raw_pvals = [0.042, 0.015, 0.08] # Mock p-values for demonstration
    _, bh_pvals, _, _ = multipletests(raw_pvals, method="fdr_bh")
    print(f"Raw p-values: {raw_pvals}\nCorrected (BH-FDR): {bh_pvals.round(4)}")

multiple_testing_correction(df_validated)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. HETEROGENEOUS TREATMENT EFFECTS (Updated for 'country')
# ─────────────────────────────────────────────────────────────────────────────
def heterogeneous_treatment_effects(df: pd.DataFrame):
    print("\n" + "=" * 60 + "\nSTEP 7: HTE (BY COUNTRY)\n" + "=" * 60)
    for country in df["country"].unique():
        seg = df[df["country"] == country]
        c_rate = seg[seg["con_treat"] == "control"]["converted"].mean()
        t_rate = seg[seg["con_treat"] == "treatment"]["converted"].mean()
        print(f"Country: {country} | Lift: {t_rate - c_rate:+.4f}")

heterogeneous_treatment_effects(df_validated)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. NOVELTY EFFECT DETECTION
# ─────────────────────────────────────────────────────────────────────────────
def novelty_effect_detection(df: pd.DataFrame):
    print("\n" + "=" * 60 + "\nSTEP 8: NOVELTY EFFECT\n" + "=" * 60)
    df = df.sort_values("time")
    mid = len(df) // 2
    early, late = df.iloc[:mid], df.iloc[mid:]
    
    early_lift = early[early["con_treat"]=="treatment"]["converted"].mean() - early[early["con_treat"]=="control"]["converted"].mean()
    late_lift = late[late["con_treat"]=="treatment"]["converted"].mean() - late[late["con_treat"]=="control"]["converted"].mean()
    
    print(f"Early Lift: {early_lift:.4f} | Late Lift: {late_lift:.4f}")
    if abs(early_lift) > abs(late_lift) * 1.5: print("⚠ Warning: Novelty effect suspected.")

novelty_effect_detection(df_validated)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. BUSINESS DECISION MEMO
# ─────────────────────────────────────────────────────────────────────────────
def generate_decision_memo(freq, bayes):
    print("\n" + "=" * 60 + "\nSTEP 9: BUSINESS MEMO\n" + "=" * 60)
    rec = "SHIP" if bayes["prob_treatment_wins"] > 0.95 else "DO NOT SHIP"
    print(f"RECOMMENDATION: {rec}\nConfidence: {bayes['prob_treatment_wins']:.1%}")

generate_decision_memo(frequentist_results, bayesian_results)

In [ ]:
def main(df):
    df = load_and_validate(df)
    design = experiment_design_doc()
    freq_results = frequentist_analysis(df)
    bayes_results = bayesian_analysis(df)
    sequential_analysis(df)
    multiple_testing_correction(df)
    heterogeneous_treatment_effects(df)
    novelty_effect_detection(df)
    generate_decision_memo(freq_results, bayes_results)
    print("\nSTEP 10: ANALYSIS COMPLETE")

main(df)